# PFAS x ACS — Data Loading & Join

Reads the two cleaned parquets from `data/raw/`, joins PFAS records to ACS ZCTA
demographics (**ZIP → ZCTA**), and writes the full merged dataset to `data/processed/`.

The two inputs are auto-detected by their columns, so filenames don't matter:
the PFAS file has `ZIP Codes Served`; the ACS file has `ZCTA`.

`ZIP Codes Served` packs multiple `"; "`-separated ZIPs, so each PFAS record is
exploded to one row per served ZIP before matching — the output grain is
**one row per (PFAS result × served ZCTA)**.

In [9]:
import os, glob
import pandas as pd
import pyarrow.parquet as pq

RAW = os.environ.get("RAW_DIR", "../data/raw")
OUT = os.environ.get("OUT_DIR", "../data/processed")
os.makedirs(OUT, exist_ok=True)

# Auto-detect which parquet is which by column signature.
pfas_path = acs_path = None
for p in sorted(glob.glob(os.path.join(RAW, "*.parquet"))):
    names = set(pq.read_schema(p).names)
    if "ZIP Codes Served" in names: pfas_path = p
    elif "ZCTA" in names:           acs_path = p
assert pfas_path, f"No PFAS parquet (with 'ZIP Codes Served') found in {RAW}"
assert acs_path,  f"No ACS parquet (with 'ZCTA') found in {RAW}"
print("PFAS:", pfas_path)
print("ACS :", acs_path)

PFAS: ../data/raw/30bb373d-f158-4d30-a06c-f729eaea6a99.parquet
ACS : ../data/raw/acs5_2023_zcta.parquet


## 1. Load PFAS (already cleaned by the conversion step)

In [10]:
pfas = pd.read_parquet(pfas_path)
print(f"{len(pfas):,} rows | detections: {int(pfas['is_detection'].fillna(False).sum()):,}"
      f" | systems: {pfas['PWS ID'].nunique():,} | contaminants: {pfas['Contaminant'].nunique()}")
pfas.head(3)

1,011,953 rows | detections: 38,698 | systems: 3,562 | contaminants: 30


,PWS ID,PWS Name,Size,Facility ID,Facility Name,Facility Water Type,Sample Point ID,Sample Point Name,Sample Point Type,Collection Date,...,Longitude,Population Served,Population Served Year,Most Recent Sample,Potential PFAS Sources,PFAS Treatment,UCMR Cycle,Results,UCMR_RAA_Active,is_detection
0,010106001,Mashantucket Pequot Water System,Large,00006,MPTN WTP,GU-Groundwater Under Influence of Surface Water,D11,WTP EPTDS,EP,2015-03-17,...,-71.972828,37807.0,2023.0,None,None,None,UCMR 3 (2013-2015),1,1.0,False
1,010106001,Mashantucket Pequot Water System,Large,00006,MPTN WTP,GU-Groundwater Under Influence of Surface Water,D11,WTP EPTDS,EP,2015-03-17,...,-71.972828,37807.0,2023.0,None,None,None,UCMR 3 (2013-2015),1,1.0,False
2,010106001,Mashantucket Pequot Water System,Large,00006,MPTN WTP,GU-Groundwater Under Influence of Surface Water,D11,WTP EPTDS,EP,2015-03-17,...,-71.972828,37807.0,2023.0,None,None,None,UCMR 3 (2013-2015),1,1.0,False


## 2. Explode `ZIP Codes Served` → one row per served ZIP (ZCTA candidate)

In [11]:
# keep only the columns the aggregation needs, before exploding (memory!)
pfas["is_detection"] = pfas["is_detection"].astype("boolean").fillna(False).astype(bool)

KEEP = ["ZIP Codes Served", "PWS ID", "Contaminant",
        "Analytical Result Value (ng/L)", "Minimum Reporting Level (ng/L)",
        "is_detection", "Collection Date"]

missing_zip = int(pfas["ZIP Codes Served"].isna().sum())
small = pfas.loc[pfas["ZIP Codes Served"].notna(), KEEP].copy()
small["zcta"] = small["ZIP Codes Served"].str.split(";")
small = small.drop(columns="ZIP Codes Served").explode("zcta")
small["zcta"] = small["zcta"].str.strip().str.zfill(5)
small = small[small["zcta"].str.fullmatch(r"\d{5}")]
small["month"] = small["Collection Date"].dt.to_period("M").dt.to_timestamp()

print(f"no-ZIP rows excluded: {missing_zip:,}")
print(f"exploded rows: {len(small):,} | unique ZCTAs served: {small['zcta'].nunique():,}")
print(f"exploded frame: {small.memory_usage(deep=True).sum()/1e6:,.0f} MB")

no-ZIP rows excluded: 28,283
exploded rows: 7,443,595 | unique ZCTAs served: 9,670
exploded frame: 1,724 MB


## 3. Load ACS and join (left join keeps every PFAS row)

In [12]:
acs = pd.read_parquet(acs_path)
acs["ZCTA"] = acs["ZCTA"].astype("string").str.zfill(5)

# aggregate to ZCTA x month FIRST, then attach ACS (so the big frame is never merged)
# small["result_filled"] = small["Analytical Result Value (ng/L)"].fillna(small["Minimum Reporting Level (ng/L)"] / 2)
pfas_agg = small.groupby(["zcta", "month"]).agg(
    mean_result_ngL=("Analytical Result Value (ng/L)", "mean"),   # mean of detected values
    # mean_result_ngL=("result_filled", "mean"),                  # <- non-detects at half-MRL
    n_records=("Analytical Result Value (ng/L)", "size"),
    n_detections=("is_detection", "sum"),
    n_systems=("PWS ID", "nunique"),
    n_contaminants=("Contaminant", "nunique"),
).reset_index()
pfas_agg["any_detection"] = pfas_agg["n_detections"] > 0

zcta_month = pfas_agg.merge(acs, how="left", left_on="zcta", right_on="ZCTA")

matched = int(zcta_month["ZCTA"].notna().sum())
print(f"ACS ZCTAs: {len(acs):,}")
print(f"{len(zcta_month):,} ZCTA-months | rows matched to ACS: {matched:,} ({matched/len(zcta_month):.1%})")
print(f"unique ZCTAs: {zcta_month['zcta'].nunique():,} | "
      f"unmatched ZCTAs: {zcta_month.loc[zcta_month['ZCTA'].isna(),'zcta'].nunique():,} "
      f"(USPS ZIPs with no ACS ZCTA -> add a ZIP->ZCTA crosswalk to recover)")
zcta_month.head()

ACS ZCTAs: 33,772
100,660 ZCTA-months | rows matched to ACS: 91,830 (91.2%)
unique ZCTAs: 9,670 | unmatched ZCTAs: 969 (USPS ZIPs with no ACS ZCTA -> add a ZIP->ZCTA crosswalk to recover)


,zcta,month,mean_result_ngL,n_records,n_detections,n_systems,n_contaminants,any_detection,ZCTA,GEO_ID,...,tenure_universe,renter_occupied,pct_below_poverty,pct_white_nh,pct_black_nh,pct_asian_nh,pct_amerind_nh,pct_hispanic,pct_nonwhite,pct_renter
0,00612,2015-03-01,NaN,42,0,1,6,False,00612,860Z200US00612,...,23438.0,9103.0,40.837773,0.397041,0.08165,0.012808,0.0,99.282764,99.602959,38.838638
1,00612,2015-10-01,NaN,42,0,1,6,False,00612,860Z200US00612,...,23438.0,9103.0,40.837773,0.397041,0.08165,0.012808,0.0,99.282764,99.602959,38.838638
2,00612,2016-11-01,NaN,36,0,1,6,False,00612,860Z200US00612,...,23438.0,9103.0,40.837773,0.397041,0.08165,0.012808,0.0,99.282764,99.602959,38.838638
3,00612,2016-12-01,NaN,24,0,1,6,False,00612,860Z200US00612,...,23438.0,9103.0,40.837773,0.397041,0.08165,0.012808,0.0,99.282764,99.602959,38.838638
4,00612,2024-04-01,NaN,319,0,1,29,False,00612,860Z200US00612,...,23438.0,9103.0,40.837773,0.397041,0.08165,0.012808,0.0,99.282764,99.602959,38.838638


## 4. Save the full merged dataset

In [13]:
out_path = os.path.join(OUT, "pfas_acs_zcta_month.parquet")
zcta_month.to_parquet(out_path, compression="zstd", index=False)
print(f"wrote {out_path}  ({os.path.getsize(out_path)/1e6:.2f} MB, "
      f"{len(zcta_month):,} ZCTA-months x {zcta_month.shape[1]} cols)")

wrote ../data/processed/pfas_acs_zcta_month.parquet  (1.95 MB, 100,660 ZCTA-months x 32 cols)


## To use the final dataset

Use:
```
import pandas as pd
df = pd.read_parquet("data/processed/pfas_acs_zcta_month.parquet")
```
